# Gold Serving Table: Daily Network KPIs

Build daily network-level operational and environmental metrics.

**Sources:**

- `workspace.urbanpulse_gold.fact_line_status`
- `workspace.urbanpulse_gold.fact_arrival_observation`
- `workspace.urbanpulse_gold.fact_weather`
- `workspace.urbanpulse_gold.dim_date`

**Target:** `workspace.urbanpulse_gold.daily_network_kpis`

**Grain:** One row per calendar date represented by at least one operational fact.

In [0]:
# Project Paths
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

In [0]:
# Imports
from pyspark.sql import functions as F

from urbanpulse.transformations.daily_network_kpis import (
    build_daily_network_kpis,
)

from urbanpulse.quality.daily_network_kpis import (
    invalid_daily_network_kpis,
)

In [0]:
# Setting up tables - sources and target
LINE_STATUS_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_line_status"
)

ARRIVAL_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_arrival_observation"
)

WEATHER_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_weather"
)

DIM_DATE_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_date"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "daily_network_kpis"
)

In [0]:
# Read Gold sources
line_status_df = spark.table(
    LINE_STATUS_TABLE
)

arrival_df = spark.table(
    ARRIVAL_TABLE
)

weather_df = spark.table(
    WEATHER_TABLE
)

dim_date_df = spark.table(
    DIM_DATE_TABLE
)

print(
    f"Line status facts: "
    f"{line_status_df.count()}"
)

print(
    f"Arrival facts: "
    f"{arrival_df.count()}"
)

print(
    f"Weather facts: "
    f"{weather_df.count()}"
)

In [0]:
# Build Daily KPIs
kpi_df = (
    build_daily_network_kpis(
        line_status_df=line_status_df,
        arrival_df=arrival_df,
        weather_df=weather_df,
        dim_date_df=dim_date_df,
    )
)

kpi_count = kpi_df.count()

print(
    f"Daily KPI rows: "
    f"{kpi_count}"
)

display(
    kpi_df
    .orderBy(
        F.col("calendar_date").desc()
    )
)

In [0]:
# Validate grain
duplicate_dates_df = (
    kpi_df
    .groupBy("date_key")
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_dates_df.count() > 0:
    display(
        duplicate_dates_df
    )

    raise ValueError(
        "Duplicate dates detected in "
        "daily_network_kpis."
    )

print(
    "Daily KPI grain validation passed."
)

In [0]:
# Apply quality checks
invalid_df = (
    invalid_daily_network_kpis(
        kpi_df
    )
)

invalid_count = invalid_df.count()

print(
    f"Invalid rows: "
    f"{invalid_count}"
)

if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"{invalid_count} invalid "
        "daily KPI rows detected."
    )

print(
    "Daily KPI quality checks passed."
)

In [0]:
# Validate line-state arithmetic
invalid_line_totals_df = (
    kpi_df
    .filter(
        F.col("line_snapshots")
        !=
        (
            F.col(
                "good_service_line_snapshots"
            )
            +
            F.col(
                "disrupted_line_snapshots"
            )
        )
    )
)

if invalid_line_totals_df.count() > 0:
    display(
        invalid_line_totals_df
    )

    raise ValueError(
        "Daily line-state totals "
        "do not reconcile."
    )

print(
    "Line-state totals reconcile."
)

In [0]:
# Add serving metadata
serving_df = (
    kpi_df
    .withColumn(
        "serving_updated_at",
        F.current_timestamp(),
    )
)

In [0]:
# Write serving table
(
    serving_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Created serving table: "
    f"{TARGET_TABLE}"
)

In [0]:
%sql
-- verify table
SELECT
    calendar_date,
    day_name,
    is_weekend,
    is_bank_holiday,
    line_snapshots,
    disrupted_line_snapshots,
    disruption_rate_pct,
    arrival_observations,
    distinct_vehicles,
    avg_eta_seconds,
    avg_temperature_c,
    total_precipitation_mm,
    max_wind_gust_kmh
FROM workspace.urbanpulse_gold.daily_network_kpis
ORDER BY calendar_date DESC;

In [0]:
%sql
-- date uniqueness check
SELECT
    date_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.daily_network_kpis
GROUP BY date_key
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Check date dimension integrity
SELECT k.*
FROM workspace.urbanpulse_gold.daily_network_kpis k

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_date d
    ON k.date_key = d.date_key;

In [0]:
%sql
-- Dashboard trend query
SELECT
    calendar_date,
    disruption_rate_pct,
    arrival_observations,
    avg_eta_seconds,
    avg_temperature_c,
    total_precipitation_mm
FROM workspace.urbanpulse_gold.daily_network_kpis
ORDER BY calendar_date;

In [0]:
%sql
-- Network Summary
SELECT
    COUNT(*) AS days_observed,

    SUM(
        line_snapshots
    ) AS line_snapshots,

    SUM(
        disrupted_line_snapshots
    ) AS disrupted_line_snapshots,

    SUM(
        arrival_observations
    ) AS arrival_observations,

    COUNT(
        DISTINCT CASE
            WHEN is_bank_holiday
            THEN date_key
        END
    ) AS bank_holiday_days

FROM workspace.urbanpulse_gold.daily_network_kpis;

In [0]:
%sql
-- Highest disruption days
SELECT
    calendar_date,
    day_name,
    is_bank_holiday,
    holiday_name,
    line_snapshots,
    disrupted_line_snapshots,
    disruption_rate_pct,
    avg_temperature_c,
    total_precipitation_mm,
    max_wind_gust_kmh
FROM workspace.urbanpulse_gold.daily_network_kpis
WHERE line_snapshots > 0
ORDER BY
    disruption_rate_pct DESC,
    calendar_date DESC;

In [0]:
%sql
-- Weather comparison query
SELECT
    CASE
        WHEN total_precipitation_mm > 0
            THEN 'Wet'
        ELSE 'Dry'
    END AS weather_group,

    COUNT(*) AS days,

    ROUND(
        AVG(disruption_rate_pct),
        2
    ) AS avg_disruption_rate_pct,

    ROUND(
        AVG(avg_eta_seconds),
        1
    ) AS avg_eta_seconds

FROM workspace.urbanpulse_gold.daily_network_kpis

WHERE weather_observations > 0

GROUP BY
    CASE
        WHEN total_precipitation_mm > 0
            THEN 'Wet'
        ELSE 'Dry'
    END;

In [0]:
%sql
-- Bank holiday comparison
SELECT
    is_bank_holiday,

    COUNT(*) AS days,

    ROUND(
        AVG(disruption_rate_pct),
        2
    ) AS avg_disruption_rate_pct,

    ROUND(
        AVG(arrival_observations),
        1
    ) AS avg_arrival_observations,

    ROUND(
        AVG(avg_eta_seconds),
        1
    ) AS avg_eta_seconds

FROM workspace.urbanpulse_gold.daily_network_kpis

GROUP BY is_bank_holiday

ORDER BY is_bank_holiday;

In [0]:
%sql
-- Indempotency check
SELECT COUNT(*) AS rows
FROM workspace.urbanpulse_gold.daily_network_kpis;